In [1]:
from glove_word_embeddings import pre,val,cat,mod
from collections import Counter, defaultdict
from nltk.corpus import wordnet as wn
import numpy as np, nltk; nltk.download("wordnet")

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/daweiwang/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Preprocessing

In [2]:
# Individual cleaning modules
print(pre.return_lowercase("  Cat! "))
print(pre.strip_marks("  Cat! "))
print(pre.strip_stopword("The Cat"))
print(pre.strip_space("jar of jam"))
print(pre.remove_stopwords("jar of jam"))
print(pre.space_check("cat"))
print(pre.space_check("jar of jam"))

  cat! 
  Cat  
Cat
jarofjam
['jar', 'jam']
True
False


In [3]:
# Full cleaning module
print(pre.clean_word("  The Cat! "))
print(pre.clean_word("jar of jam"))
print(pre.clean_word("jar of jam", strip_space_bool=False))
print(pre.clean_word("the", strip_stopword_bool=False))

cat
jarjam
jar jam
the


In [4]:
# WordNet lemmatization module
print(pre.lemmatize("dogs"))
print(pre.lemmatize("running", pos="v"))

dog
run


### Validation

In [5]:
# WordNet validations
print(val.word("telescope"))
print(val.word("jar of jam"))
print(val.noun("telescope"))
print(val.noun("quickly"))

True
False
True
False


In [6]:
# Olson et al., 2021's wordlist
m = mod.load("glove-olson-validated")
v = m.vocab_set()
print(val.vocab("telescope", v))
print(val.vocab("ice cream", v))

True
False


### Categorization (Bucket)

In [7]:
# Ordinary common word?
print(cat.check_common("apple"))
print(cat.check_common("nike"))
print(cat.check_common("aaron"))

True
False
False


In [8]:
# Which curated buckets does a word belong to?
print(cat.check_bucket("dog"))
print(cat.check_bucket("paris", check_common=False))
print(cat.check_bucket("apple", check_common=True))
print(cat.check_bucket("apple", check_common=False))

{'animals'}
{'places'}
{'plants', 'fruits'}
{'plants', 'brands', 'fruits'}


In [9]:
# Count across a list of words
words = ['test',                    # 1 one-word subjects
         'dog','cat',               # 2 one-word animals
         'apple',                   # Brand and object
         'chair','table','desk',    # 3 one-word objects         
         'sun glasses',             # 2 compound
         'wr0ng','w0rd']            # 2 wrong words

print(cat.count_bucket(words, 3))
print(cat.count_bucket(words, 4))
print(cat.count_bucket(words, name="environment", number_of_words=1, check_common=True))
print(cat.count_bucket(words, name="animals", number_of_words=1, check_common=True))
print(cat.count_bucket(words, name="brands", number_of_words=1, check_common=False))
print(cat.count_bucket(words, name="places", number_of_words=1, check_common=True))
print(cat.count_bucket(words, name="names",  number_of_words=1, check_common=True))
print(cat.count_bucket(words, name="brands", number_of_words=1, check_common=True))

True
False
True
True
True
False
False
False


### Categorization (WordNet)

In [10]:
depth_counts = Counter()
unique_names = defaultdict(set)

for synset in wn.all_synsets(pos="n"):
    d = synset.min_depth()               # shortest path length to entity
    depth_counts[d] += 1
    name = synset.name().split(".")[0]   # e.g. "dog" instead of "dog.n.01"
    unique_names[d].add(name)

print("Noun categories at each level (min_depth from entity):\n")
print(f"{'Level':>6}  {'Synsets':>8}  {'Unique names':>12}")
print("-" * 32)
for level in sorted(depth_counts):
    print(f"{level:6d}  {depth_counts[level]:8d}  {len(unique_names[level]):12d}")

print(f"Total noun synsets : {sum(depth_counts.values())}")
print(f"Maximum depth      : {max(depth_counts)}")

Noun categories at each level (min_depth from entity):

 Level   Synsets  Unique names
--------------------------------
     0         1             1
     1         3             3
     2        22            22
     3       228           216
     4      2020          1918
     5      6249          5790
     6     12267         11129
     7     18936         17636
     8     14155         13412
     9     11042         10539
    10      7207          6993
    11      4267          4174
    12      2505          2453
    13      1383          1366
    14       846           838
    15       449           443
    16       341           338
    17       164           158
    18        30            30
Total noun synsets : 82115
Maximum depth      : 18


In [11]:
def synset_depth(ss):
    """Minimum depth from the root (entity.n.01)."""
    return min(len(p) for p in ss.hypernym_paths())

# ----- 1. Depth distribution of all noun synsets -----
depths = []
for ss in wn.all_synsets("n"):
    depths.append(synset_depth(ss))

depths = np.array(depths)
print("Total noun synsets:", len(depths))
print("Max depth         :", depths.max())

print("Synsets at each exact depth:")
cnt = Counter(depths)
for d in range(1, int(depths.max()) + 1):
    print(f"  level {d-1:2d}: {cnt[d]:6d}")

print("Cumulative – fraction that reach at least depth k:")
for k in range(0, 12):
    frac = (depths >= k).mean()
    print(f"  ≥ {k:2d}: {frac:.3f}  ({(depths >= k).sum()} synsets)")

# ----- 2. Is every branch able to grow to level 8? -----
# (i.e. does every synset that exists at level 7 have at least one hyponym path that continues?)
reach_8 = (depths >= 8).sum()
print(f"Synsets that reach depth 8 or deeper: {reach_8}")
print(f"Fraction of all noun synsets that reach ≥ 8: {(depths >= 8).mean():.3f}")

# ----- 3. Density comparison -----
print("Density (number of synsets) around the peak:")
for d in range(5, 11):
    print(f"  level {d}: {cnt[d]}")

Total noun synsets: 82115
Max depth         : 19
Synsets at each exact depth:
  level  0:      1
  level  1:      3
  level  2:     22
  level  3:    228
  level  4:   2020
  level  5:   6249
  level  6:  12267
  level  7:  18936
  level  8:  14155
  level  9:  11042
  level 10:   7207
  level 11:   4267
  level 12:   2505
  level 13:   1383
  level 14:    846
  level 15:    449
  level 16:    341
  level 17:    164
  level 18:     30
Cumulative – fraction that reach at least depth k:
  ≥  0: 1.000  (82115 synsets)
  ≥  1: 1.000  (82115 synsets)
  ≥  2: 1.000  (82114 synsets)
  ≥  3: 1.000  (82111 synsets)
  ≥  4: 1.000  (82089 synsets)
  ≥  5: 0.997  (81861 synsets)
  ≥  6: 0.972  (79841 synsets)
  ≥  7: 0.896  (73592 synsets)
  ≥  8: 0.747  (61325 synsets)
  ≥  9: 0.516  (42389 synsets)
  ≥ 10: 0.344  (28234 synsets)
  ≥ 11: 0.209  (17192 synsets)
Synsets that reach depth 8 or deeper: 61325
Fraction of all noun synsets that reach ≥ 8: 0.747
Density (number of synsets) around the peak

In [12]:
# Full ladder (general → specific)
print('fullest primary:',cat.category_chain("badger", path='full', sense='primary'))
print('shortest primary:',cat.category_chain("badger", path='short', sense='primary'))
print('longest primary:',cat.category_chain("badger", path='long', sense='primary'))
print('fullest union:',cat.category_chain("badger", path='full', sense='union'))
print('shortest union:',cat.category_chain("badger", path='short', sense='union'))
print('longest union:',cat.category_chain("badger", path='long', sense='union'))

# Single rung (0 = most general)
print(cat.category_by_level("dog", level=0))
print(cat.category_by_level("dog", level=4))

# Do two words share the same category at a given level?
print(cat.check_same_category("dog", "wolf"))
print(cat.check_same_category("dog", "wolf", level=1))

# Most specific shared category name / its level on word1’s chain
print(cat.category_shared_name("dog", "wolf"))
print(cat.category_shared_level("dog", "wolf"))

fullest primary: ['entity', 'physical_entity', 'causal_agent', 'object', 'whole', 'living_thing', 'organism', 'person', 'inhabitant', 'american']
shortest primary: ['entity', 'physical_entity', 'causal_agent', 'person', 'inhabitant', 'american']
longest primary: ['entity', 'physical_entity', 'object', 'whole', 'living_thing', 'organism', 'person', 'inhabitant', 'american']
fullest union: ['entity', 'physical_entity', 'causal_agent', 'object', 'whole', 'living_thing', 'organism', 'animal', 'person', 'chordate', 'inhabitant', 'american', 'vertebrate', 'mammal', 'placental', 'carnivore', 'musteline_mammal']
shortest union: ['entity', 'physical_entity', 'causal_agent', 'person', 'inhabitant', 'american']
longest union: ['entity', 'physical_entity', 'object', 'whole', 'living_thing', 'organism', 'animal', 'chordate', 'vertebrate', 'mammal', 'placental', 'carnivore', 'musteline_mammal']
entity
living_thing
True
True
canine
12


### Embedding

In [21]:
m = mod.load("glove-6b-300d")

print(m.embed_exact("cat").shape)
print(len(m.vocab_set()),'\n')
print(m.embed_phrase("jar of jam")[:36])

(300,)
400000 

[-0.05391501 -0.16789001 -0.339425   -0.479775    0.53409505 -0.470175
  0.18782501  0.02561501 -0.166271    0.036045   -0.2297005   0.206965
 -0.228724    0.42599     0.24285498  0.41358    -0.37821    -0.01148
 -0.02718001  0.38415998  0.45273     0.042023   -0.04648501  0.1745605
 -0.091915   -0.35143    -0.05492    -0.459495    0.421205   -0.043835
 -0.316745    0.1279875   0.148793   -0.176585   -0.196497    0.49551   ]


In [22]:
m = mod.load("tasa-lsa-300d")

print(m.embed_exact("cat").shape)
print(len(m.vocab_set()),'\n')
print(m.embed_phrase("jar of jam")[:36])

(300,)
92393 

[ 0.00273909  0.00208881 -0.00419593 -0.00322939  0.00137018  0.00056892
 -0.00278634  0.00208912  0.00117144 -0.00201013  0.00154777  0.00277286
  0.002856   -0.00163077 -0.00203997 -0.00219596 -0.00160106 -0.00598526
  0.00195464 -0.00015027  0.00146301 -0.00254607  0.00186757 -0.00536568
  0.00138295  0.00012608 -0.00158022 -0.0009371   0.00321231 -0.00235425
  0.00073752  0.00509192  0.00106857  0.00152289  0.00511054 -0.00570042]


In [23]:
m = mod.load("bert-large")

print(m.embed_exact("cat").shape)
print(len(m.vocab_set()),'\n')
print(m.embed_phrase("jar of jam")[:36])

(1024,)
30522 

[-0.09700519 -0.16091879  0.86675423 -0.19233376 -0.01102528 -0.15819144
  0.01325796  0.19714542 -0.10745873  0.6354454  -0.07352979 -0.14711158
 -0.18632261 -0.02118639 -1.0104694   0.7622168  -0.02152689  0.16403885
  0.5320373   0.31565168  1.1239114  -0.56436914 -0.2224557  -0.15861271
 -0.3610785  -0.2631361   0.6495853   0.6676876   0.20634298 -0.01239076
 -0.84154433  0.18394077  0.45776877 -0.01543576  0.8173515   0.3738546 ]


In [24]:
m = mod.load("bert-base")

print(m.embed_exact("cat").shape)
print(len(m.vocab_set()),'\n')
print(m.embed_phrase("jar of jam")[:36])

(768,)
30522 

[ 0.8111565   0.31777775  0.46894005  0.26425943  0.9881219  -0.37986556
  0.488376   -0.6945035  -0.9877302  -0.00293623 -0.9676234  -0.02477086
  0.8477452   0.50620335  0.32422724  1.074584   -0.01391434  0.78996515
 -0.04357306  1.1302428   0.23545305 -0.7024811  -0.9784166   0.8697465
  0.1729825  -0.3913579  -0.34573016  0.22804356  0.35050774 -0.8149521
  0.80275375  0.20199616  0.09175157 -0.01013798 -0.87765884 -0.46780333]
